In [1]:
!pip install geopy pandas tqdm

  Using cached geopy-2.4.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached geographiclib-2.1-py3-none-any.whl.metadata (1.6 kB)
Using cached geopy-2.4.1-py3-none-any.whl (125 kB)
Using cached geographiclib-2.1-py3-none-any.whl (40 kB)

   ---------------------------------------- 0/2 [geographiclib]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]




[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import geopandas as gpd
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

In [21]:
gdf = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Santa Maria\Padronização\CVS_PRONTO\EEE_v0.gpkg')

In [22]:
gdf

,ETAPA_CICLO,REGIONAL,UF,UN,MUNICIPIO,BAIRRO,BACIA,NOTAS,DESCRICAO,ELEVACAO_TERRENO,...,CREATED_USER,CREATED_DATE,LAST_EDITED_USER,LAST_EDITED_DATE,SUPERINTENDENCIA,GRUPO_EXECUTIVO,SUB_BACIA,DISTRITO,NOME,geometry
0,Projeto Conceitual,None,None,None,None,None,SB-21I,None,Em andamento,NaN,...,None,NaT,None,NaT,None,None,SB-21I,None,SB-21I,POINT Z (223418.576 6708448.546 0)
1,Projeto Conceitual,None,None,None,None,None,SB-21H,None,Em andamento,NaN,...,None,NaT,None,NaT,None,None,SB-21H,None,SB-21H,POINT Z (223123.977 6709310.839 0)
2,Projeto Conceitual,None,None,None,None,None,SB-21F,None,Em andamento,NaN,...,None,NaT,None,NaT,None,None,SB-21F,None,SB-21F,POINT Z (223065.817 6709648.929 0)
3,Projeto Conceitual,None,None,None,None,None,SB-21D,None,Em andamento,NaN,...,None,NaT,None,NaT,None,None,SB-21D,None,SB-21D,POINT Z (223745.096 6709796.933 0)
4,Projeto Conceitual,None,None,None,None,None,SB-21C-2,None,Em andamento,NaN,...,None,NaT,None,NaT,None,None,SB-21C-2,None,SB-21C-2,POINT Z (223798.937 6709935.443 0)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,Projeto Conceitual,None,None,None,None,None,SB-23B-1,None,Em andamento,NaN,...,None,NaT,None,NaT,None,None,SB-23B-1,None,SB-23B-1,POINT Z (227216.897 6709916.031 0)
87,Projeto Conceitual,None,None,None,None,None,SB-EC57,None,Proposicao EC,NaN,...,None,NaT,None,NaT,None,None,SB-EC57,None,SB-EC57,POINT Z (233185.506 6710329.899 0)
88,Projeto Conceitual,None,None,None,None,None,SB-EC58,None,Proposicao EC,NaN,...,None,NaT,None,NaT,None,None,SB-EC58,None,SB-EC58,POINT Z (233491.683 6708924.698 0)
89,Projeto Conceitual,None,None,None,None,None,SB-33D,None,Em andamento,NaN,...,None,NaT,None,NaT,None,None,SB-33D,None,SB-33D,POINT Z (226877.532 6714821.396 0)


In [24]:
# ----------------------------
# 1) GARANTIR CRS UTM 22S
#    (se seu gdf já tem crs correto, pode pular o set_crs)
# ----------------------------
# Ex.: SIRGAS2000 / UTM 22S -> EPSG:31982
# Ex.: WGS84 / UTM 22S      -> EPSG:32722
# gdf = ...  # seu GeoDataFrame de pontos
# if gdf.crs is None:
#     gdf = gdf.set_crs(epsg=31982)  # ajuste para 32722 se for o seu caso

# ----------------------------
# 2) REPROJETAR PARA WGS84
# ----------------------------
gdf_wgs = gdf.to_crs(epsg=4326).copy()
gdf_wgs["lat"] = gdf_wgs.geometry.y
gdf_wgs["lon"] = gdf_wgs.geometry.x

# ----------------------------
# 3) CONFIGURAR NOMINATIM + RATE LIMIT
# ----------------------------
geolocator = Nominatim(user_agent="bairro_osm_extractor")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)  # gentil com a API

# ----------------------------
# 4) FUNÇÃO PARA EXTRAIR BAIRRO
# ----------------------------
def extrair_bairro(lat, lon):
    try:
        loc = reverse((lat, lon), language="pt")
        if not loc or "address" not in loc.raw:
            return None
        addr = loc.raw["address"]
        # ordem de preferência dos campos que costumam representar "bairro"
        return (
            addr.get("suburb")
            or addr.get("neighbourhood")
            or addr.get("city_district")
            or addr.get("quarter")
            or addr.get("village")
            or addr.get("town")
            or addr.get("city")
        )
    except Exception:
        return None

# ----------------------------
# 5) CACHE: evita repetir chamadas para as mesmas coords
#    (arredonda p/ reduzir duplicatas quase idênticas)
# ----------------------------
def chave_cache(lat, lon, casas=5):
    return (round(lat, casas), round(lon, casas))

coords = gdf_wgs[["lat", "lon"]].copy()
coords["key"] = coords.apply(lambda r: chave_cache(r["lat"], r["lon"]), axis=1)

# Deduplica
coords_uniq = coords.drop_duplicates("key").reset_index(drop=True)

# Consulta com barra de progresso
tqdm.pandas(desc="Buscando bairros (OSM)")
coords_uniq["bairro"] = coords_uniq.progress_apply(
    lambda r: extrair_bairro(r["lat"], r["lon"]), axis=1
)

# Mapeia de volta para todas as linhas
mapa_bairros = dict(zip(coords_uniq["key"], coords_uniq["bairro"]))
gdf_wgs["bairro"] = coords["key"].map(mapa_bairros)

# ----------------------------
# 6) (Opcional) ANEXAR AO GDF ORIGINAL
# ----------------------------
# Se você quer manter o CRS UTM e apenas adicionar a coluna:
gdf_final = gdf.copy()
gdf_final["BAIRRO"] = gdf_wgs["bairro"]

# Resultado:
# gdf_final tem a sua geometria original (UTM 22S) + coluna 'bairro'
print(gdf_final[["BAIRRO", "geometry"]].head())


Buscando bairros (OSM): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 91/91 [01:37<00:00,  1.07s/it]

      BAIRRO                            geometry
0  Boi Morto  POINT Z (223418.576 6708448.546 0)
1  Boi Morto  POINT Z (223123.977 6709310.839 0)
2  Boi Morto  POINT Z (223065.817 6709648.929 0)
3  Boi Morto  POINT Z (223745.096 6709796.933 0)
4  Boi Morto  POINT Z (223798.937 6709935.443 0)


In [25]:
# salva seu GeoDataFrame em um arquivo .gpkg
gdf_final.to_file(r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Santa Maria\Padronização\CVS_PRONTO\EEE_v1.gpkg", driver="GPKG")

In [26]:
def extrair_cota_com_fallback(ponto, src_principal, src_fallback=None, vetor_crs="EPSG:31982"):
    """
    Extrai a cota (elevação) de um ponto a partir do MDE principal.
    Se não houver valor válido ou ocorrer erro, tenta obter a cota em um MDE de fallback.

    Argumentos:
        ponto (shapely.geometry.Point ou MultiPoint): ponto a ser avaliado (no CRS dos vetores).
        src_principal (rasterio.DatasetReader): MDE principal aberto.
        src_fallback (rasterio.DatasetReader | None): MDE de fallback (baixa resolução), se houver.
        vetor_crs (str ou CRS): CRS dos vetores (ex.: 'EPSG:31981').

    Retorna:
        float: cota extraída ou np.nan em caso de falha.
    """
    if ponto is None or ponto.is_empty:
        return np.nan

    try:
        # Coordenadas no CRS dos vetores
        if ponto.geom_type == 'MultiPoint':
            x, y = ponto.geoms[0].x, ponto.geoms[0].y
        elif ponto.geom_type == 'Point':
            x, y = ponto.x, ponto.y
        else:
            return np.nan  # Tipo não suportado

        # Reprojetar para o CRS do raster principal
        transformer = Transformer.from_crs(vetor_crs, src_principal.crs, always_xy=True)
        x_dem, y_dem = transformer.transform(x, y)

        row, col = src_principal.index(x_dem, y_dem)
        valor = src_principal.read(1, window=((row, row+1), (col, col+1)))[0, 0]

        # Se nodata ou NaN, tenta fallback
        if (hasattr(src_principal, "nodata") and valor == src_principal.nodata) or np.isnan(valor):
            if src_fallback is not None:
                return extrair_cota_dem_fallback(ponto, src_fallback, vetor_crs=vetor_crs)
            return np.nan

        return float(valor)

    except Exception:
        # Se der erro, tenta diretamente no fallback (se existir)
        if src_fallback is not None:
            try:
                return extrair_cota_dem_fallback(ponto, src_fallback, vetor_crs=vetor_crs)
            except Exception:
                return np.nan
        return np.nan


def extrair_cota_dem_fallback(ponto, src_fallback, vetor_crs="EPSG:31982"):
    """
    Extrai a cota (elevação) de um ponto a partir do MDE de fallback (baixa resolução).

    Argumentos:
        ponto (shapely.geometry.Point ou MultiPoint): ponto a ser avaliado (no CRS dos vetores).
        src_fallback (rasterio.DatasetReader): MDE de fallback aberto.
        vetor_crs (str ou CRS): CRS dos vetores.

    Retorna:
        float: cota extraída ou np.nan em caso de falha.
    """
    if ponto is None or ponto.is_empty:
        return np.nan

    try:
        # Coordenadas no CRS dos vetores
        if ponto.geom_type == 'MultiPoint':
            x, y = ponto.geoms[0].x, ponto.geoms[0].y
        elif ponto.geom_type == 'Point':
            x, y = ponto.x, ponto.y
        else:
            return np.nan  # Tipo não suportado

        # Reprojetar para o CRS do raster de fallback
        transformer = Transformer.from_crs(vetor_crs, src_fallback.crs, always_xy=True)
        x_dem, y_dem = transformer.transform(x, y)

        row, col = src_fallback.index(x_dem, y_dem)
        valor = src_fallback.read(1, window=((row, row+1), (col, col+1)))[0, 0]

        if (hasattr(src_fallback, "nodata") and valor == src_fallback.nodata) or np.isnan(valor):
            return np.nan

        return float(valor)

    except Exception:
        return np.nan

In [14]:
import os
import numpy as np
import geopandas as gpd
import rasterio
from pyproj import Transformer

In [27]:
def preencher_cotas_em_gpkg(
    gpkg_in: str,
    dem_principal: str,
    gpkg_out: str,
    dem_fallback: str | None = None,
    layer: str | None = None,
    coluna_cota: str = "ELEVACAO_TERRENO",
    vetor_crs_padrao: str = "EPSG:31982",
    overwrite: bool = True,
):
    """
    Lê um GPKG (pontos), extrai cota do DEM principal (com fallback opcional),
    grava um novo GPKG com a coluna de cota preenchida.

    - Se 'layer' for None, lê a primeira layer do GPKG.
    - 'vetor_crs_padrao' só é usado se o arquivo não tiver CRS definido.
    """

    if overwrite and os.path.exists(gpkg_out):
        os.remove(gpkg_out)

    # Descobrir layer se não veio informado
    if layer is None:
        layers = gpd.list_layers(gpkg_in)
        if len(layers) == 0:
            raise ValueError("Nenhuma layer encontrada no GPKG de entrada.")
        layer = layers.iloc[0]["name"]

    gdf = gpd.read_file(gpkg_in, layer=layer)

    if gdf.empty:
        raise ValueError("A layer está vazia.")

    # Garantir CRS
    if gdf.crs is None:
        gdf = gdf.set_crs(vetor_crs_padrao, allow_override=True)
        vetor_crs = vetor_crs_padrao
    else:
        vetor_crs = gdf.crs

    # Garantir que a coluna existe
    if coluna_cota not in gdf.columns:
        gdf[coluna_cota] = np.nan

    # Abrir rasters e calcular
    with rasterio.open(dem_principal) as src_principal:
        if dem_fallback:
            with rasterio.open(dem_fallback) as src_fb:
                gdf[coluna_cota] = gdf.geometry.apply(
                    lambda geom: extrair_cota_com_fallback(
                        geom, src_principal, src_fallback=src_fb, vetor_crs=vetor_crs
                    )
                )
        else:
            gdf[coluna_cota] = gdf.geometry.apply(
                lambda geom: extrair_cota_com_fallback(
                    geom, src_principal, src_fallback=None, vetor_crs=vetor_crs
                )
            )

    # Exportar mantendo atributos
    gdf.to_file(gpkg_out, layer=layer, driver="GPKG")

    # Retorno útil (opcional)
    total = len(gdf)
    preenchidos = int(np.isfinite(gdf[coluna_cota]).sum())
    return {"layer": layer, "total": total, "preenchidos": preenchidos, "saida": gpkg_out}


# ==========================
# EXEMPLO DE USO
# ==========================
if __name__ == "__main__":
    resultado = preencher_cotas_em_gpkg(
        gpkg_in=r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Santa Maria\Padronização\CVS_PRONTO\EEE_v1.gpkg",
        dem_principal=r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Santa Maria\tiff.tif",
        gpkg_out=r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Santa Maria\Padronização\CVS_PRONTO\E_EEE.gpkg",
        dem_fallback=r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Santa Maria\tiff.tif",  # ou None
        layer=None,  # ou "nome_da_layer"
        coluna_cota="ELEVACAO_TERRENO",
        vetor_crs_padrao="EPSG:31982",
        overwrite=True,
    )
    print(resultado)

{'layer': 'EEE_v1', 'total': 91, 'preenchidos': 91, 'saida': 'C:\\Users\\gabriel.coimbra\\Desktop\\Meus arquivos\\Santa Maria\\Padronização\\CVS_PRONTO\\E_EEE.gpkg'}
